
# Lab Test - Agentic RAG on Union Budget 2026-27 Speech

Name: Prajakta Ghorpade
Roll No.: SCFP125004
Document: Union Budget 2026–2027 Speech  

This notebook uses document ingestion, text chunking, Hugging Face embeddings, Chroma vector search, and a tool-using GenAI agent to answer questions from the Budget Speech.

## Setup

In [ ]:
!pip install langchain-core langchain-text-splitters langchain-huggingface langchain-chroma chromadb "langchain-docling[local]" langsmith

In [ ]:

!pip install -qU langchain-groq

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")

## Q1 - Ingestion

In [ ]:
from langchain_docling.loader import DoclingLoader

FILE_PATH = "/kaggle/input/budget-speech/budget_speech.pdf"

loader = DoclingLoader(file_path=FILE_PATH)
documents = loader.load()

In [ ]:
len(documents)

Find the text-extraction problem (count > 0 is the problem)

In [ ]:
import re

problems = {
    "rupee symbol as backtick": r"`",
    "unmapped glyph (cid)": r"\(cid:\d+\)",
    "broken ligature": r"[ﬁﬂﬀﬃﬄ]",
    "hyphen split word": r"\w-\n\w",
    "extra spaces": r"[ \t]{2,}",
}

for name, pattern in problems.items():
    count = sum(len(re.findall(pattern, doc.page_content)) for doc in documents)
    print(name, "->", count)

Fix the problem

In [ ]:
def fix_text(text):
    text = re.sub(r"`\s?(?=\d)", "Rs. ", text)
    text = re.sub(r"\(cid:\d+\)", "", text)
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text

for doc in documents:
    doc.page_content = fix_text(doc.page_content)

for name, pattern in problems.items():
    count = sum(len(re.findall(pattern, doc.page_content)) for doc in documents)
    print(name, "->", count)

Keep page number (copy it from `dl_meta` into a simple `page` key)

In [ ]:
def get_page(doc):
    pages = []
    for item in doc.metadata["dl_meta"]["doc_items"]:
        for prov in item["prov"]:
            pages.append(prov["page_no"])
    return min(pages) if pages else 0

for doc in documents:
    doc.metadata["page"] = get_page(doc)

Split into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
texts = text_splitter.split_documents(documents)

In [ ]:
print(len(texts))
print(texts[0].metadata["page"])
print(texts[0].page_content)

## Q2 - Vector Store

In [ ]:
from langchain_core.documents import Document

def clean_metadata(doc: Document) -> Document:
    simple = {k: v for k, v in doc.metadata.items() if isinstance(v, (str, int, float, bool))}
    return Document(page_content=doc.page_content, metadata=simple)

clean_texts = [clean_metadata(d) for d in texts]

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Build and persist

In [ ]:
import shutil
from langchain_chroma import Chroma

persist_dir = "/kaggle/working/chroma_budget_db"
shutil.rmtree(persist_dir, ignore_errors=True)

vector_store = Chroma.from_documents(
    documents=clean_texts,
    embedding=embeddings,
    collection_name="budget_speech",
    persist_directory=persist_dir
)

Reload from the persist directory

In [ ]:
vector_store = Chroma(
    collection_name="budget_speech",
    embedding_function=embeddings,
    persist_directory=persist_dir
)

Top 3 results with scores (Chroma score is distance, lower = better)

In [ ]:
results = vector_store.similarity_search_with_score("Biopharma SHAKTI outlay", k=3)

for doc, score in results:
    print("score:", score, "| page:", doc.metadata["page"])
    print(doc.page_content)
    print("-" * 50)

## Q3 - Search Tool

In [ ]:
from langchain_core.tools import tool

@tool
def search_budget(query: str) -> str:
    """Search the Union Budget 2026-27 speech document. Use it for any question about budget schemes, outlays, numbers, duties, taxes or policies. Returns the matching text with page numbers."""
    docs = vector_store.similarity_search(query, k=4)
    result = ""
    for doc in docs:
        result += f"[Page {doc.metadata['page']}] {doc.page_content}\n\n"
    return result

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

llm_with_tools = llm.bind_tools([search_budget])
tool_map = {"search_budget": search_budget}

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM_PROMPT = """You are a helpful assistant for the Union Budget 2026-27 speech document.
You have a tool search_budget that searches the document.

Rules:
1. For any question about facts, numbers, schemes, duties, taxes or policies of the budget, call search_budget first. Do not answer these from memory.
2. For greetings, small talk and general concept explanations (example: what is fiscal deficit), answer directly without calling any tool.
3. When your answer uses the document, cite the page number like (Page 12) after each fact.
4. If the search results do not contain the answer, say exactly: The document does not contain this information. Do not guess and do not use outside knowledge for budget facts.
5. Keep the answer short and clear."""

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history")
])

In [ ]:

from langchain_core.messages import HumanMessage, ToolMessage

def ask_agent(user_query):
    history = [HumanMessage(content=user_query)]
    tools_used = []

    ai_msg = llm_with_tools.invoke(agent_prompt.invoke({"history": history}))
    history.append(ai_msg)

    for call in ai_msg.tool_calls:
        tools_used.append(call["name"])
        tool_result = tool_map[call["name"]].invoke(call["args"])
        history.append(ToolMessage(content=str(tool_result), tool_call_id=call["id"]))

    if ai_msg.tool_calls:
        ai_msg = llm_with_tools.invoke(agent_prompt.invoke({"history": history}))

    print("Tools used:", tools_used)
    return ai_msg.content

Model searches

In [ ]:
print(ask_agent("What is the outlay of Biopharma SHAKTI?"))

Model does not search

In [ ]:
print(ask_agent("Hi, how are you?"))

Answer not in document

In [ ]:
print(ask_agent("What is the budget allocation for the Mars colonisation mission?"))

## Q4 - Calculator Tool

In [ ]:
import ast
import operator

allowed_ops = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    elif isinstance(node, ast.BinOp) and type(node.op) in allowed_ops:
        left = safe_eval(node.left)
        right = safe_eval(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("power is too large")
        return allowed_ops[type(node.op)](left, right)
    elif isinstance(node, ast.UnaryOp) and type(node.op) in allowed_ops:
        return allowed_ops[type(node.op)](safe_eval(node.operand))
    else:
        raise ValueError("only numbers and + - * / % ** are allowed")

@tool
def calculator(expression: str) -> str:
    """Safe calculator. Pass one arithmetic expression using numbers and + - * / % ** and brackets. Example: (5000 - 4000) / 4000 * 100. Use it for every sum, difference and percentage."""
    try:
        tree = ast.parse(expression.replace(",", ""), mode="eval")
        return str(round(safe_eval(tree.body), 4))
    except Exception as e:
        return f"Error: {e}"

In [ ]:
llm_with_tools = llm.bind_tools([search_budget, calculator])
tool_map = {"search_budget": search_budget, "calculator": calculator}

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant for the Union Budget 2026-27 speech document.
You have two tools:
- search_budget : searches the document
- calculator    : safe arithmetic

Rules:
1. For any question about facts, numbers, schemes, duties, taxes or policies of the budget, call search_budget first. Do not answer these from memory.
2. For greetings, small talk and general concept explanations (example: what is fiscal deficit), answer directly without calling any tool.
3. For any sum, difference, ratio or percentage, first find the numbers with search_budget, then use calculator. Never calculate in your head.
4. You can call tools many times. Search separately for each number you need.
5. When your answer uses the document, cite the page number like (Page 12) after each fact. For a calculated value, cite the pages of the numbers used.
6. If the search results do not contain the answer, say exactly: The document does not contain this information. Do not guess and do not use outside knowledge for budget facts.
7. Keep the answer short and clear."""

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history")
])

Agent loop with maximum rounds

In [ ]:
def run_agent(history, max_rounds=5):
    tools_used = []

    for round_no in range(max_rounds):
        ai_msg = llm_with_tools.invoke(agent_prompt.invoke({"history": history}))
        history.append(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg.content, tools_used

        for call in ai_msg.tool_calls:
            tools_used.append(call["name"])
            tool_result = tool_map[call["name"]].invoke(call["args"])
            history.append(ToolMessage(content=str(tool_result), tool_call_id=call["id"]))

 
    return "Stopped: maximum rounds reached without a final answer.", tools_used

Interactive chat loop (ends when user types `exit`)

In [ ]:
chat_log = []

def chat_loop(input_fn=input, max_rounds=5):
    history = []
    while True:
        try:
            user_query = input_fn("Enter user query: ")
        except (EOFError, RuntimeError):
            print("No input available, stopping the loop")
            break
        if user_query.strip().lower() == "exit":
            print("Bye")
            break

        history.append(HumanMessage(content=user_query))
        answer, tools_used = run_agent(history, max_rounds)

        print("Tools used:", tools_used)
        print("Answer:", answer)
        print("-" * 50)

        chat_log.append({
            "query": user_query,
            "searched": "Y" if "search_budget" in tools_used else "N",
            "tools_used": ", ".join(tools_used) if tools_used else "none",
            "final_answer": answer
        })

## Q5 - Test Queries

The 8 queries are given to the same `chat_loop` one by one (Save & Run All cannot type). The last input is `exit`.

In [ ]:
test_queries = [
    "What is the outlay of Biopharma SHAKTI?",
    "How does the 2026-27 fiscal deficit compare with 2025-26?",
    "What changes for NCCD on chewing tobacco, and what is the effective duty?",
    "Hi, how are you?",
    "Explain fiscal deficit in simple words.",
    "What are the new income tax slab rates in this budget?",
    "By what percentage does public capex rise from BE 2025-26 to FY2026-27?",
    "What is the combined outlay of Biopharma SHAKTI, the Container Manufacturing scheme and CCUS?",
    "exit"
]

query_iter = iter(test_queries)

def scripted_input(prompt):
    user_query = next(query_iter)
    print(prompt + user_query)
    return user_query

chat_loop(input_fn=scripted_input)

Table: query, searched (Y/N), tools used, final answer

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

result_df = pd.DataFrame(chat_log)
result_df

Decision accuracy (Q4 and Q5 should not search, all others should search)

In [ ]:
expected_search = ["Y", "Y", "Y", "N", "N", "Y", "Y", "Y"]

result_df["expected"] = expected_search[:len(result_df)]
result_df["correct"] = result_df["searched"] == result_df["expected"]

correct = int(result_df["correct"].sum())
print(f"Decision accuracy: {correct}/{len(result_df)} = {correct / len(result_df) * 100:.1f}%")

print("Wrong decisions:")
for _, row in result_df[~result_df["correct"]].iterrows():
    print("-", row["query"], "| searched:", row["searched"], "| expected:", row["expected"])

### Explanation of wrong decision (write after seeing your output)

Accuracy: ___ / 8

Wrong decision: Query ___ . The model searched / did not search, but it should have ___ .
Reason: ___